<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/CST2312_H11/blob/main/CST2312_Class12_NestedDictionaries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CST2312 – Data and Information Management I
## Class 12: Nested Dictionaries and Inventory Management
**Professor Patrick** | New York City College of Technology
*Textbook: [Foundations of Python Programming (FOPP)](https://runestone.academy/ns/books/published/fopp/index.html)*  
  
<i>a copy of this notebook is available at [bit.ly/sneaker_dictionary](https://bit.ly/sneaker_dictionary) </i>

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Construct** a nested dictionary from structured string data
2. **Navigate** and **retrieve** values from a two-level nested dictionary using compound key access
3. **Parse** a CSV-formatted docstring into a list of dictionaries
4. **Validate** user input using `try/except` and string normalization
5. **Filter** a dataset by date to compute a point-in-time inventory state
6. **Apply** defensive programming techniques to handle missing keys and invalid inventory quantities
7. **Compute** derived fields (Inventory Value) from dictionary data
8. **Display** a formatted tabular report using Python print statements

## FOPP Section Reference

This notebook draws on the following sections of *Foundations of Python Programming*:

| FOPP Section | Topic | Relevance to This Notebook |
|---|---|---|
| Ch. 11 — Dictionaries | Creating and accessing dictionaries | Foundation for all steps |
| Ch. 11.4 — Dictionary Methods | `.get()`, `.keys()`, `.items()` | Key access patterns throughout |
| Ch. 11.7 — Nested Dictionaries | Two-level key structures | Core data structure of the exercise |
| Ch. 9.1–9.3 — Strings | `.strip()`, `.split()`, `.lower()` | Parsing the CSV docstrings |
| Ch. 7.3 — `try` / `except` | Exception handling | User input and edge-case handling |

---

## Session Plan

| Section | Topic | Estimated Time |
|---|---|---|
| Background | Nested dictionaries — concepts and motivation | 5 min |
| Step 1 | Parse inventory data and construct the nested dictionary | 15 min |
| Step 2 | Parse transaction data into a list of records | 10 min |
| Step 3 | Collect and validate a target date from the user | 10 min |
| Step 4 | Filter and apply transactions with defensive error handling | 20 min |
| Step 5 | Compute inventory value and print the final report | 10 min |
| **Total** | | **70 min** |

---

## Background: Nested Dictionaries

### What Is a Nested Dictionary?

A Python dictionary maps each **key** to a **value**. In a *nested* dictionary, the value associated with an outer key is itself a dictionary. This two-level structure is a natural representation for real-world records where each entity (outer key) carries multiple named attributes (inner keys).

**General form:**

```python
nested_dict = {
    "outer_key_1": {
        "field_a": value,
        "field_b": value,
    },
    "outer_key_2": {
        "field_a": value,
        "field_b": value,
    },
}
```

### Accessing Values

To retrieve a specific inner value, chain both keys in sequence:

```python
nested_dict["outer_key_1"]["field_a"]
```

To update an inner value in place:

```python
nested_dict["outer_key_1"]["field_a"] = new_value
```

---

### Application: Shoe Boutique Inventory

#### Business Context

**Sole & Co.** is a sneaker and streetwear boutique located in Lower Manhattan. The store carries a curated selection of Nike, Adidas, and Vans footwear. Like all retail businesses, Sole & Co. must track the quantity of each product-size combination it holds at any given time — information collectively known as its **inventory**.

December is the most consequential month in the retail calendar. Holiday shopping drives a surge in both sales volume and restocking activity. For the store manager, understanding exactly how inventory levels and total inventory value are shifting week by week — or even day by day — during this period is essential for making sound business decisions: when to reorder stock, which sizes are selling fastest, and whether the overall value of goods on hand is trending in the expected direction.

#### The Business Problem

The store manager has asked for a reporting tool that can answer the following question at any point during December:

> *"As of a given date in December 2026, what is the current count on hand and total inventory value for each SKU in our catalog?"*

A **SKU** (stock-keeping unit) is the unique identifier the store uses for each product-size combination. For example, `NK-AF1-090` identifies a Nike Air Force 1 Low in size 9.0, while `AD-SMB-100` identifies an Adidas Samba in size 10.0. Because a size 10.0 and a size 11.0 of the same model are distinct sellable units, they carry distinct SKUs.

The manager's question requires:
- A **starting inventory** recorded as of December 1, 2026 (the state of the stockroom at the opening of the month)
- A **transaction log** recording every purchase (incoming stock) and sale (outgoing stock) that occurred during December
- A mechanism to **apply transactions selectively** up to any chosen date, so that the resulting inventory reflects exactly what was on hand at the close of business on that day

#### Why a Nested Dictionary?

The inventory naturally fits a nested dictionary structure. Each SKU is a unique identifier — the ideal **outer key**. Each SKU has multiple associated attributes — brand, style, size, cost per unit, and count on hand — which are stored as the **inner dictionary**. This structure supports fast, direct lookups by SKU and makes it straightforward to read or update any individual attribute.

| Inner Field | Description | Example |
|---|---|---|
| `brand` | Manufacturer name | `"Nike"` |
| `style` | Model name | `"Air Force 1 Low"` |
| `size` | Shoe size — stored as a categorical string | `"10.5"` |
| `cost` | Retail cost per unit in dollars | `120` |
| `count` | Units currently on hand | `8` |

The exercise proceeds through five steps, each building directly on the previous one.

---

## Imports

Run the cell below before executing any other cell in this notebook.

In [ ]:
# Standard library modules only — no additional installation required
from datetime import datetime   # Used in Step 3 and Step 4 for date parsing and comparison

---
## Step 1 — Load the Inventory Data and Construct the Nested Dictionary
*(Estimated time: 15 minutes)*

### Context

The starting inventory for Sole & Co. is available in two equivalent forms — you may use whichever approach your environment supports:

**Option A — Docstring (recommended for Colab without file upload)**
The variable `INVENTORY_DATA` is defined in **Appendix A** at the bottom of this notebook as a CSV-formatted multi-line string. Run the Appendix A cell first, then use `INVENTORY_DATA.strip().splitlines()` to obtain a list of lines.

**Option B — CSV file**
The inventory data is also available as a CSV file hosted in Professor Patrick’s GitHub data repository. You may download it directly from the link below and upload it to your Colab session, or save it to your local machine.

> 🔗 **Inventory data file:** [sneaker_inventory.csv](https://bit.ly/sneaker_inventory)
> Download from Professor Patrick’s GitHub ‘data’ repository. To use in Colab, download the file and upload it via the file browser panel on the left-hand side of the interface, then run the code below.

> 🔗 **Transaction data file (for Step 2):** [sneaker_transactions.csv](https://bit.ly/sneaker_transactions)
> Download from Professor Patrick’s GitHub ‘data’ repository. Upload to Colab using the same procedure before running Step 2.

Once `sneaker_inventory.csv` has been uploaded to your Colab session, you may read it using Python’s built-in `open()` function:

```python
with open("sneaker_inventory.csv", "r") as f:
    lines = f.read().strip().splitlines()
```

Both options produce an identical list of lines. All subsequent code in this step is the same regardless of which option you choose.

---

Each row in the data (after the header) represents one SKU and contains the following comma-separated fields:

```
SKU, Brand, Style, Size, Cost_Per_Unit, Count_On_Hand
```

**Example row:**
```
NK-AF1-090, Nike, Air Force 1 Low, 9.0, 120, 8
```

### Your Task

Write code to:
1. Load the inventory data using Option A or Option B above to obtain a list of lines named `lines`
2. Skip the header row and any blank lines
3. Parse each data row by splitting on commas and stripping whitespace
4. Construct a nested dictionary named `inventory` where:
   - The **outer key** is the SKU string
   - The **inner dictionary** contains keys: `brand`, `style`, `size`, `cost`, `count`
   - `cost` should be stored as an `int`
   - `count` should be stored as an `int`

> **Note on field types:** `size` is treated as a categorical string (e.g., `"10.5"`) rather than a float. This reflects common retail practice where sizes are labels, not numeric quantities.

In [ ]:
# ── Step 1: Load inventory data and construct the nested dictionary ──────────

inventory = {}   # Initialize the outer dictionary

# ── Data Source Selection ─────────────────────────────────────────────────────
# Choose ONE of the two options below and comment out the other.

# OPTION A — Docstring (run the Appendix A cell first)
lines = INVENTORY_DATA.strip().splitlines()

# OPTION B — CSV file (upload sneaker_inventory.csv to your Colab session first)
# with open("sneaker_inventory.csv", "r") as f:
#     lines = f.read().strip().splitlines()

# ─────────────────────────────────────────────────────────────────────────────
# Both options above produce an identical 'lines' list.
# Complete the steps below regardless of which option you chose.
# ─────────────────────────────────────────────────────────────────────────────

# TODO 1: Iterate over lines[1:], skipping the header and any blank lines.
#         For each data line:
#           a. Split the line on ',' to get a list of fields
#           b. Strip whitespace from each field using a list comprehension
#           c. Unpack fields into named variables:
#              sku, brand, style, size, raw_cost, raw_count
#           d. Build the inner dictionary and assign it to inventory[sku]

for line in lines[1:]:          # [1:] skips the header row
    if not line.strip():
        continue                # skip blank lines

    fields = None               # TODO 1a: split on ','
    fields = None               # TODO 1b: strip each field

    # TODO 1c: unpack fields into named variables
    sku      = None
    brand    = None
    style    = None
    size     = None
    raw_cost = None
    raw_count= None

    # TODO 1d: store the inner dictionary in inventory
    inventory[sku] = None       # replace None with the inner dictionary

In [ ]:
# ── Step 1 Verification ───────────────────────────────────────────────────────
# Run this cell after completing Step 1 to check your work.

assert isinstance(inventory, dict),                       "inventory must be a dict"
assert len(inventory) == 44,                              f"Expected 44 SKUs, found {len(inventory)}"
assert "NK-AF1-090" in inventory,                         "SKU NK-AF1-090 is missing"
assert inventory["NK-AF1-090"]["brand"] == "Nike",        "Brand mismatch for NK-AF1-090"
assert inventory["NK-AF1-090"]["style"] == "Air Force 1 Low", "Style mismatch for NK-AF1-090"
assert inventory["NK-AF1-090"]["size"]  == "9.0",         "Size mismatch for NK-AF1-090"
assert isinstance(inventory["NK-AF1-090"]["cost"],  int), "cost must be an int"
assert isinstance(inventory["NK-AF1-090"]["count"], int), "count must be an int"
assert inventory["NK-AJ11-095"]["cost"] == 200,           "AJ11 cost should be 200"

print("Step 1 verification passed.  Inventory dictionary contains",
      len(inventory), "SKUs.")

# Preview the first three entries
print("\nSample entries:")
for sku in list(inventory)[:3]:
    print(f"  {sku}: {inventory[sku]}")

---
## Step 2 — Load the Transaction Data into a List of Records
*(Estimated time: 10 minutes)*

### Context

Transaction data for December 2026 is available in the same two forms as the inventory data:

**Option A — Docstring**
The variable `TRANSACTION_DATA` is defined in **Appendix B** at the bottom of this notebook. Run the Appendix B cell first.

**Option B — CSV file**
If you have uploaded `sneaker_transactions.csv` to your Colab session, read it with:

```python
with open("sneaker_transactions.csv", "r") as f:
    tx_lines = f.read().strip().splitlines()
```

---

Each row represents one inventory event:

```
SKU, Date, Transaction_Type, Quantity, Notes
```

| Field | Description | Example |
|---|---|---|
| `SKU` | Identifies the product-size combination | `"NK-DNK-100"` |
| `Date` | Transaction date in `YYYY-MM-DD` format | `"2026-12-05"` |
| `Transaction_Type` | Either `Purchased` or `Sold` | `"Sold"` |
| `Quantity` | Number of units involved | `3` |
| `Notes` | Free-text annotation — present in the data but **not used in any calculation** | `"Holiday season restock"` |

> **Note on the Notes field:** Real-world transaction datasets routinely contain fields that are relevant for auditing or record-keeping but play no role in a given analysis. The `Notes` field here is a deliberate example of such a distractor. A data scientist must read the field descriptions carefully and consciously decide which fields to use — and which to carry along without acting on them.

### Your Task

Produce a list named `transactions` where each element is a dictionary with keys: `sku`, `date`, `type`, `quantity`, `notes`.

- `date` should be stored as a `datetime` object (use `datetime.strptime`)
- `quantity` should be stored as an `int`
- All other fields should be stored as strings

In [ ]:
# ── Step 2: Load transaction data into a list of record dictionaries ─────────

transactions = []   # Initialize the list

# ── Data Source Selection ─────────────────────────────────────────────────────
# Choose ONE of the two options below and comment out the other.

# OPTION A — Docstring (run the Appendix B cell first)
tx_lines = TRANSACTION_DATA.strip().splitlines()

# OPTION B — CSV file (upload sneaker_transactions.csv to your Colab session first)
# with open("sneaker_transactions.csv", "r") as f:
#     tx_lines = f.read().strip().splitlines()

# ─────────────────────────────────────────────────────────────────────────────

# TODO 1: Iterate over tx_lines[1:], skipping blank lines.
#         For each data line:
#           a. Split on ',' using maxsplit=4 to preserve any commas in the Notes field,
#              then strip whitespace from each field
#           b. Parse the date string using datetime.strptime(date_str, "%Y-%m-%d")
#           c. Append a dictionary to transactions with keys:
#              'sku', 'date', 'type', 'quantity', 'notes'

for line in tx_lines[1:]:
    if not line.strip():
        continue

    fields = None               # TODO 1a: split with maxsplit=4, then strip
    fields = None

    raw_sku, raw_date, raw_type, raw_qty, raw_notes = None, None, None, None, None

    # TODO 1b: parse the date string
    parsed_date = None

    # TODO 1c: append the record dictionary
    transactions.append(None)

In [ ]:
# ── Step 2 Verification ───────────────────────────────────────────────────────
assert isinstance(transactions, list),                    "transactions must be a list"
assert len(transactions) == 150,                          f"Expected 150 records, found {len(transactions)}"
assert isinstance(transactions[0], dict),                 "Each record must be a dict"
assert isinstance(transactions[0]["date"], datetime),     "date must be a datetime object"
assert isinstance(transactions[0]["quantity"], int),      "quantity must be an int"

# Check date range
dates = [t["date"] for t in transactions]
print(f"Step 2 verification passed.")
print(f"  Total records : {len(transactions)}")
print(f"  Earliest date : {min(dates).strftime('%Y-%m-%d')}")
print(f"  Latest date   : {max(dates).strftime('%Y-%m-%d')}")
print(f"  Sold          : {sum(1 for t in transactions if t['type']=='Sold')}")
print(f"  Purchased     : {sum(1 for t in transactions if t['type']=='Purchased')}")

---
## Data Context: What the Inventory and Transaction Data Represent

Before proceeding to Step 3, it is important to understand precisely what the data loaded in Steps 1 and 2 represents — and what question the remaining steps are designed to answer.

### Starting Inventory: As of December 1, 2026

The `inventory` dictionary you constructed in Step 1 captures the state of Sole & Co.'s stockroom **at the opening of business on December 1, 2026**. Every count in that dictionary reflects how many units of each SKU were physically on the shelf at that moment — before any December transaction had taken place.

This is the **baseline** from which all subsequent calculations proceed.

### Transaction Log: All of December 2026

The `transactions` list you built in Step 2 contains **150 inventory events** spanning December 1 through December 30, 2026. Each record represents one of two things:

- A **Purchased** transaction: the store received new stock from a supplier, increasing the count on hand for that SKU
- A **Sold** transaction: the store sold one or more units to a customer, decreasing the count on hand

Together, these two datasets allow you to reconstruct the exact state of the inventory **at any point during December** by starting from the December 1 baseline and applying only those transactions that occurred on or before the chosen date.

### The Query: Any Date in December

The reporting tool you are building is designed to answer the manager's question for **any date the user specifies**. For example:

| Query Date | What the Report Shows |
|---|---|
| December 1, 2026 | Baseline inventory — no transactions applied |
| December 15, 2026 | Inventory after the first two weeks of holiday shopping |
| December 30, 2026 | Near-end-of-month state, reflecting nearly all holiday activity |

The date-filtering logic in Step 4 is what makes this flexibility possible. By including only transactions with a date **on or before** the target date, the snapshot accurately reflects what was on hand at the close of business that day.

> **Important constraint:** The valid query range is **December 1 through December 30, 2026**. Dates outside this range will produce results that do not reflect real business activity — either the unmodified baseline (for dates before December 1) or an incomplete picture (for dates after December 30, since the dataset does not extend beyond that date).

---

---
## Step 3 — Collect and Validate a Target Date from the User
*(Estimated time: 10 minutes)*

### Context

The goal of this exercise is to compute inventory levels **as of a specific date** — that is, the inventory state after all transactions on or before the target date have been applied.  This requires collecting a date from the user and validating that it is in the correct format before any calculations are performed.

### Input Specification

- Use Python's built-in `input()` function to prompt the user
- The expected format is `YYYY-MM-DD` (e.g., `2026-12-15`)
- Normalize the input using `.strip().lower()` before attempting to parse it
- Wrap the parsing attempt in a `try/except` block
  - If parsing succeeds, store the result as a `datetime` object named `target_date`
  - If parsing fails (the user entered an unrecognizable format), print a clear error message and set `target_date = None`

> **Discussion question:** Why is `.lower()` applied here even though dates do not contain letters?  Consider what characters a user might inadvertently type, and how normalization habits protect against a broader class of input errors.

In [ ]:
# ── Step 3: Collect and validate the target date from the user ───────────────

target_date = None   # Will hold a datetime object if input is valid

# TODO 1: Use input() to prompt the user for a date string.
#         Normalize the raw input using .strip().lower() and store it.
raw_input = None

# TODO 2: Wrap the following in a try/except block.
#         Inside try:  parse raw_input using datetime.strptime with format "%Y-%m-%d"
#                      and assign to target_date
#         Inside except ValueError: print a descriptive error message
#                      and leave target_date as None

In [ ]:
# ── Step 3 Verification ───────────────────────────────────────────────────────
# This check only runs if target_date was successfully set.
if target_date is not None:
    assert isinstance(target_date, datetime), "target_date must be a datetime object"
    print(f"Step 3 verification passed.  target_date = {target_date.strftime('%Y-%m-%d')}")
else:
    print("target_date is None — re-run Step 3 and enter a valid date before continuing.")

---
## Step 4 — Filter Transactions and Apply Them to the Inventory Dictionary
*(Estimated time: 20 minutes)*

### Context

With a validated `target_date` in hand, you are ready to compute the inventory state as of that date.  This requires iterating over the `transactions` list, selecting only those records with a date on or before `target_date`, and updating the `count` field in the `inventory` dictionary accordingly.

Before writing the main loop, you must create a **working copy** of the inventory.  Modifying the original `inventory` dictionary directly would make it impossible to re-run this step for a different date without restarting from Step 1.  Use a deep copy so that the original data is preserved.

### Two Edge Cases to Handle

Real transaction data is imperfect.  This dataset contains two deliberate errors that your code must detect and handle gracefully using `try/except`:

**Edge Case 1 — Unrecognized SKU**
A transaction references a SKU that does not exist in the inventory dictionary.  Attempting to access `inventory[sku]` when `sku` is not a key will raise a `KeyError`.  Catch this exception, print a warning that identifies the offending SKU and transaction date, and continue processing the remaining transactions.

**Edge Case 2 — Negative Inventory**
A `Sold` transaction reduces `count` below zero.  This represents a data entry error (selling more units than are on hand).  After applying the transaction, check whether `count` has gone negative.  If it has, print a warning identifying the SKU, the transaction date, and the resulting count, and reset `count` to `0`.

### Logic for Transaction Types

- `Purchased`: increase `count` by `quantity`
- `Sold`: decrease `count` by `quantity`

> **Note on copy depth:** Python's `dict.copy()` produces a *shallow* copy — sufficient for the outer dictionary, but the inner dictionaries remain shared references.  To safely copy a nested dictionary, use a dictionary comprehension that explicitly copies each inner dictionary.

In [ ]:
# ── Step 4: Apply filtered transactions to a working copy of the inventory ────
#
# Prerequisite: target_date must not be None (see Step 3).

# TODO 1: Create 'inventory_snapshot' as a deep copy of 'inventory'.
#         Use a dictionary comprehension:
#         {sku: dict(inner) for sku, inner in inventory.items()}
inventory_snapshot = None

# TODO 2: Initialize two counters to track edge cases encountered:
#         'unrecognized_sku_count' and 'negative_inventory_count'
unrecognized_sku_count  = None
negative_inventory_count = None

# TODO 3: Iterate over the transactions list.
#         For each transaction:
#           a. Skip records where transaction['date'] > target_date
#           b. Retrieve the transaction's sku, type, and quantity
#           c. Wrap the update logic in try/except KeyError to handle Edge Case 1
#           d. Inside try: apply the Purchased/Sold logic to inventory_snapshot[sku]["count"]
#           e. After updating count, check for Edge Case 2 (negative count)
#              If count < 0: print a warning, reset count to 0, increment counter
#           f. In except KeyError: print a warning, increment unrecognized_sku_count

for txn in transactions:

    # TODO 3a: date filter
    if None:
        continue

    # TODO 3b
    sku      = None
    txn_type = None
    qty      = None

    # TODO 3c / 3d / 3e / 3f
    try:
        if txn_type == "Purchased":
            pass   # TODO: increment count
        elif txn_type == "Sold":
            pass   # TODO: decrement count

        # TODO 3e: check for negative inventory after update
        if None:
            print(f"  WARNING: Negative inventory after Sold transaction — "
                  f"SKU {sku} on {txn['date'].strftime('%Y-%m-%d')}: "
                  f"count = {inventory_snapshot[sku]['count']}.  Reset to 0.")
            inventory_snapshot[sku]["count"] = 0
            negative_inventory_count += 1

    except KeyError:
        # TODO 3f
        print(f"  WARNING: Unrecognized SKU '{sku}' in transaction dated "
              f"{txn['date'].strftime('%Y-%m-%d')}.  Skipped.")
        unrecognized_sku_count += 1

print(f"\nTransaction processing complete.")
print(f"  Unrecognized SKU warnings  : {unrecognized_sku_count}")
print(f"  Negative inventory warnings: {negative_inventory_count}")

In [ ]:
# ── Step 4 Verification ───────────────────────────────────────────────────────
assert inventory_snapshot is not inventory,               "snapshot must be a separate object"
assert len(inventory_snapshot) == 44,                     "snapshot must still contain 44 SKUs"
# Confirm original inventory was not modified
assert all(inventory[s]["count"] == inventory[s]["count"]
           for s in inventory),                           "original inventory should be unchanged"
assert unrecognized_sku_count  >= 1,                      "expected at least one unrecognized SKU warning"
assert negative_inventory_count >= 1,                     "expected at least one negative inventory warning"
print("Step 4 verification passed.")
print(f"  inventory_snapshot contains {len(inventory_snapshot)} SKUs.")

---
## Step 5 — Compute Inventory Value and Print the Final Report
*(Estimated time: 10 minutes)*

### Context

The final output is a formatted tabular report printed to the console.  It should display the state of the inventory as of `target_date`, with one row per SKU and a summary line at the bottom.

### Output Specification

**Header line:**

```
SKU             Brand      Style                    Size    Cost    Count   Value
```

**Data rows** (one per SKU, sorted by SKU):

```
NK-AF1-090      Nike       Air Force 1 Low          9.0     $120    5       $600
```

**Summary line** (after all rows):

```
────────────────────────────────────────────────────────────────────────────────
TOTAL INVENTORY VALUE AS OF 2026-12-15:                             $XX,XXX
```

### Formatting Notes

- Use tab characters (`\t`) to separate columns
- Prefix `cost` and per-SKU `value` with `$`
- Format the total value with comma-separated thousands: use Python's f-string format spec `{value:,}` (e.g., `$12,480`)
- Sort the output rows by SKU using Python's built-in `sorted()` function

### Computed Field

**Inventory Value** for each SKU = `cost` × `count` (using values from `inventory_snapshot`)

In [ ]:
# ── Step 5: Print the formatted inventory report ─────────────────────────────

# TODO 1: Print a report title that includes the target_date
#         Example: "Inventory Report — as of December 15, 2026"
print(None)

# TODO 2: Print the column header line using tab-separated labels
print(None)

# TODO 3: Print a separator line (a string of '-' repeated ~80 times)
print(None)

# TODO 4: Initialize a variable 'total_value' to 0

total_value = None

# TODO 5: Iterate over inventory_snapshot.items() sorted by SKU.
#         For each SKU:
#           a. Retrieve brand, style, size, cost, count from the inner dict
#           b. Compute sku_value = cost * count
#           c. Add sku_value to total_value
#           d. Print a tab-separated row:
#              sku, brand, style, size, $cost, count, $sku_value

for sku, info in sorted(None):      # TODO 5: replace None with inventory_snapshot.items()
    brand    = None
    style    = None
    size     = None
    cost     = None
    count    = None
    sku_value= None                 # TODO 5b

    total_value += None             # TODO 5c

    print(None)                     # TODO 5d

# TODO 6: Print the separator line again and the total value summary
print(None)
print(None)

In [ ]:
# ── Step 5 Verification ───────────────────────────────────────────────────────
assert isinstance(total_value, int),   "total_value must be an integer"
assert total_value > 0,                "total_value should be positive"
print(f"Step 5 verification passed.  Total inventory value = ${total_value:,}")

---
## Glossary

| Term | Definition |
|---|---|
| **Nested dictionary** | A dictionary in which one or more values are themselves dictionaries, creating a two-level key-value hierarchy |
| **SKU (Stock-Keeping Unit)** | A unique alphanumeric identifier assigned to a specific product-variant combination in a retail inventory system |
| **Docstring** | A string literal that appears as the first statement of a module, function, class, or code block; used here as a convenient container for inline data |
| **`try` / `except`** | A Python construct for handling runtime exceptions; the `try` block contains code that may raise an error, and the `except` block specifies how to respond if a particular error type occurs |
| **`KeyError`** | An exception raised when a dictionary is accessed with a key that does not exist |
| **Deep copy** | A copy of a data structure in which nested objects are also duplicated, rather than shared by reference with the original |
| **Categorical field** | A data field whose values represent discrete labels or categories rather than numeric quantities; shoe size is treated categorically in this exercise |
| **Point-in-time inventory** | The state of an inventory as of a specific date, computed by applying all transactions up to and including that date |
| **`datetime.strptime()`** | A method that parses a date/time string into a `datetime` object according to a specified format string |

---
## Reading Assignment

Before Class 13, review the following sections in *Foundations of Python Programming*:

- **Chapter 11.7** — Nested Dictionaries (re-read with the exercise in mind)
- **Chapter 7** — Exceptions (`try` / `except` in greater depth)

---
## Appendix — Data Cells

The two cells below define the CSV-formatted docstrings used as **Option A** in Steps 1 and 2.  **Run both cells before running any of the exercise steps.**

> These cells are placed in the Appendix so that the narrative flow of the exercise is not interrupted by raw data.  Students who prefer to read from external files may use **Option B** in Steps 1 and 2, which reads from `sneaker_inventory.csv` and `sneaker_transactions.csv` respectively.  Both files are available for download alongside this notebook.  To use them in Google Colab, upload the file via the file browser panel on the left-hand side of the Colab interface before running the relevant step.

### Appendix A — Inventory Data

In [ ]:
INVENTORY_DATA = """
SKU,Brand,Style,Size,Cost_Per_Unit,Count_On_Hand
NK-AF1-090,Nike,Air Force 1 Low,9.0,120,14
NK-AF1-095,Nike,Air Force 1 Low,9.5,120,5
NK-AF1-100,Nike,Air Force 1 Low,10.0,120,4
NK-AF1-105,Nike,Air Force 1 Low,10.5,120,15
NK-AF1-110,Nike,Air Force 1 Low,11.0,120,8
NK-AF1-120,Nike,Air Force 1 Low,12.0,120,7
NK-DNK-090,Nike,Dunk Low,9.0,130,7
NK-DNK-095,Nike,Dunk Low,9.5,130,6
NK-DNK-100,Nike,Dunk Low,10.0,130,15
NK-DNK-110,Nike,Dunk Low,11.0,130,5
NK-DNK-115,Nike,Dunk Low,11.5,130,14
NK-DNK-130,Nike,Dunk Low,13.0,130,15
NK-AJ11-095,Nike,Air Jordan 11,9.5,200,18
NK-AJ11-100,Nike,Air Jordan 11,10.0,200,12
NK-AJ11-105,Nike,Air Jordan 11,10.5,200,5
NK-AJ11-110,Nike,Air Jordan 11,11.0,200,13
NK-AJ11-120,Nike,Air Jordan 11,12.0,200,10
NK-CTZ-090,Nike,Cortez,9.0,90,4
NK-CTZ-095,Nike,Cortez,9.5,90,4
NK-CTZ-100,Nike,Cortez,10.0,90,5
NK-CTZ-105,Nike,Cortez,10.5,90,7
NK-CTZ-110,Nike,Cortez,11.0,90,7
AD-SMB-090,Adidas,Samba,9.0,110,12
AD-SMB-095,Adidas,Samba,9.5,110,13
AD-SMB-100,Adidas,Samba,10.0,110,4
AD-SMB-105,Adidas,Samba,10.5,110,12
AD-SMB-110,Adidas,Samba,11.0,110,7
AD-SMB-115,Adidas,Samba,11.5,110,15
AD-CMP-090,Adidas,Campus 00s,9.0,100,14
AD-CMP-100,Adidas,Campus 00s,10.0,100,15
AD-CMP-105,Adidas,Campus 00s,10.5,100,12
AD-CMP-110,Adidas,Campus 00s,11.0,100,10
AD-CMP-120,Adidas,Campus 00s,12.0,100,7
AD-CLC-095,Adidas,Climacool,9.5,130,11
AD-CLC-100,Adidas,Climacool,10.0,130,13
AD-CLC-105,Adidas,Climacool,10.5,130,8
AD-CLC-110,Adidas,Climacool,11.0,130,16
AD-CLC-130,Adidas,Climacool,13.0,130,17
VN-OSK-090,Vans,Old Skool,9.0,70,4
VN-OSK-095,Vans,Old Skool,9.5,70,16
VN-OSK-100,Vans,Old Skool,10.0,70,16
VN-OSK-105,Vans,Old Skool,10.5,70,6
VN-OSK-110,Vans,Old Skool,11.0,70,15
VN-OSK-120,Vans,Old Skool,12.0,70,10
"""

### Appendix B — Transaction Data

In [ ]:
TRANSACTION_DATA = """
SKU,Date,Transaction_Type,Quantity,Notes
AD-CLC-095,2026-12-01,Purchased,12,Staff discount applied
NK-DNK-090,2026-12-01,Purchased,7,Online order fulfillment
NK-AF1-120,2026-12-02,Purchased,7,Online order fulfillment
VN-OSK-110,2026-12-02,Purchased,8,Shipment arrived with minor box damage
NK-DNK-095,2026-12-02,Sold,3,Walk-in customer purchase
NK-DNK-095,2026-12-03,Purchased,8,Gift purchase with receipt
AD-CLC-110,2026-12-03,Sold,1,Rush order from supplier
NK-AF1-120,2026-12-03,Sold,2,Gift purchase with receipt
NK-AF1-100,2026-12-04,Sold,2,Checked against PO #4821
NK-CTZ-100,2026-12-04,Sold,2,Return processed separately
AD-CMP-110,2026-12-04,Purchased,6,Weekend sale transaction
NK-CTZ-105,2026-12-05,Sold,1,Staff discount applied
NK-AF1-095,2026-12-06,Purchased,7,Verified by manager on duty
AD-SMB-090,2026-12-06,Sold,3,Rush order from supplier
NK-AJ11-110,2026-12-08,Purchased,5,Customer exchange processed
AD-CMP-120,2026-12-08,Sold,1,Customer exchange processed
NK-CTZ-110,2026-12-09,Sold,3,Gift purchase with receipt
VN-OSK-120,2026-12-09,Purchased,10,Shipment arrived with minor box damage
NK-AJ11-110,2026-12-09,Sold,1,Special order item
AD-CMP-120,2026-12-09,Sold,1,Inventory audit adjustment
VN-OSK-090,2026-12-10,Sold,3,Gift purchase with receipt
AD-CMP-100,2026-12-10,Sold,3,Shipped from warehouse
AD-SMB-100,2026-12-11,Sold,2,Checked against PO #4821
NK-DNK-100,2026-12-11,Sold,2,Walk-in customer purchase
AD-CMP-105,2026-12-11,Sold,3,Holiday season restock
NK-AF1-105,2026-12-12,Sold,2,Shipment arrived with minor box damage
NK-CTZ-100,2026-12-12,Purchased,5,Shipped from warehouse
AD-CLC-100,2026-12-13,Purchased,6,Staff discount applied
VN-OSK-110,2026-12-14,Sold,3,Online order fulfillment
NK-AJ11-120,2026-12-14,Sold,1,Awaiting customer pickup
NK-AJ11-095,2026-12-14,Sold,2,Online order fulfillment
NK-AJ11-120,2026-12-15,Sold,3,Awaiting customer pickup
NK-CTZ-090,2026-12-15,Sold,1,Vendor delivery confirmed
NK-AJ11-110,2026-12-15,Sold,3,Shipment arrived with minor box damage
AD-SMB-115,2026-12-15,Sold,3,Loyalty member transaction
NK-SB-110,2026-12-15,Sold,1,Walk-in customer purchase
AD-SMB-095,2026-12-16,Sold,3,Loyalty member transaction
AD-CMP-100,2026-12-16,Sold,3,Awaiting customer pickup
VN-OSK-120,2026-12-16,Sold,1,Return processed separately
AD-SMB-100,2026-12-16,Purchased,4,Return processed separately
NK-DNK-090,2026-12-16,Sold,1,Weekend sale transaction
NK-CTZ-100,2026-12-16,Purchased,4,Shipped from warehouse
AD-SMB-110,2026-12-17,Sold,3,Vendor delivery confirmed
AD-SMB-100,2026-12-18,Sold,1,Walk-in customer purchase
NK-AF1-110,2026-12-18,Sold,3,Gift purchase with receipt
VN-OSK-120,2026-12-19,Sold,2,Phone order placed earlier
NK-AF1-110,2026-12-19,Sold,2,Awaiting customer pickup
AD-CMP-100,2026-12-19,Purchased,9,Loyalty member transaction
AD-SMB-090,2026-12-19,Purchased,9,Loyalty member transaction
AD-CMP-105,2026-12-19,Sold,3,Phone order placed earlier
NK-CTZ-095,2026-12-20,Purchased,7,Shipment arrived with minor box damage
VN-OSK-120,2026-12-20,Sold,1,Gift purchase with receipt
NK-AF1-090,2026-12-20,Sold,3,Holiday season restock
NK-AF1-110,2026-12-20,Purchased,9,Customer exchange processed
NK-AJ11-095,2026-12-20,Purchased,11,Rush order from supplier
NK-AF1-100,2026-12-21,Purchased,12,Shipment arrived with minor box damage
AD-CLC-105,2026-12-21,Purchased,4,Shipment arrived with minor box damage
AD-CLC-095,2026-12-21,Purchased,10,Staff discount applied
NK-AJ11-095,2026-12-21,Sold,2,Walk-in customer purchase
AD-CMP-100,2026-12-21,Sold,3,Awaiting customer pickup
AD-CLC-130,2026-12-21,Sold,3,Customer exchange processed
NK-AJ11-100,2026-12-21,Purchased,9,Gift purchase with receipt
AD-SMB-105,2026-12-21,Sold,3,Requested by floor supervisor
NK-DNK-115,2026-12-21,Purchased,12,Inventory audit adjustment
NK-AF1-100,2026-12-21,Sold,2,Rush order from supplier
VN-OSK-120,2026-12-21,Sold,1,Walk-in customer purchase
AD-SMB-100,2026-12-21,Sold,2,Vendor delivery confirmed
AD-SMB-110,2026-12-21,Purchased,7,Awaiting customer pickup
AD-CMP-105,2026-12-21,Sold,2,Special order item
NK-AJ11-105,2026-12-21,Sold,1,Vendor delivery confirmed
AD-SMB-115,2026-12-22,Sold,1,Loyalty member transaction
NK-AJ11-100,2026-12-22,Sold,3,Checked against PO #4821
AD-CMP-120,2026-12-22,Purchased,9,Customer exchange processed
NK-AJ11-110,2026-12-22,Sold,2,Awaiting customer pickup
AD-SMB-115,2026-12-22,Sold,2,Vendor delivery confirmed
NK-CTZ-105,2026-12-22,Purchased,5,Loyalty member transaction
AD-CLC-105,2026-12-22,Sold,1,Customer exchange processed
NK-DNK-090,2026-12-22,Purchased,8,Gift purchase with receipt
AD-SMB-115,2026-12-22,Sold,5,End of day batch entry — quantity unverified
NK-AJ11-100,2026-12-23,Purchased,10,Special order item
NK-AJ11-105,2026-12-23,Sold,3,Staff discount applied
AD-SMB-115,2026-12-23,Sold,3,Shipment arrived with minor box damage
VN-OSK-110,2026-12-23,Purchased,8,Phone order placed earlier
VN-OSK-110,2026-12-23,Sold,2,Requested by floor supervisor
AD-SMB-105,2026-12-23,Sold,1,Requested by floor supervisor
AD-CMP-100,2026-12-23,Sold,3,Staff discount applied
AD-CMP-090,2026-12-23,Sold,3,Phone order placed earlier
AD-SMB-100,2026-12-23,Purchased,10,Awaiting customer pickup
VN-OSK-110,2026-12-23,Sold,2,Awaiting customer pickup
NK-CTZ-090,2026-12-24,Sold,2,Shipment arrived with minor box damage
AD-SMB-115,2026-12-24,Sold,3,Holiday season restock
NK-AJ11-100,2026-12-24,Purchased,5,Rush order from supplier
VN-OSK-105,2026-12-24,Sold,3,Awaiting customer pickup
NK-AF1-095,2026-12-24,Sold,2,Online order fulfillment
AD-SMB-100,2026-12-24,Sold,2,Loyalty member transaction
NK-AJ11-105,2026-12-24,Purchased,9,Customer exchange processed
AD-SMB-090,2026-12-24,Purchased,8,Requested by floor supervisor
NK-AF1-120,2026-12-25,Sold,3,Checked against PO #4821
AD-CLC-110,2026-12-25,Sold,1,Holiday season restock
NK-CTZ-090,2026-12-25,Purchased,7,Rush order from supplier
VN-OSK-090,2026-12-25,Purchased,7,Phone order placed earlier
AD-CLC-110,2026-12-25,Sold,1,Inventory audit adjustment
AD-SMB-115,2026-12-25,Sold,3,Vendor delivery confirmed
NK-DNK-115,2026-12-25,Sold,3,Shipment arrived with minor box damage
AD-SMB-095,2026-12-26,Sold,1,Phone order placed earlier
AD-CMP-110,2026-12-26,Sold,2,Holiday season restock
VN-OSK-110,2026-12-26,Sold,3,Vendor delivery confirmed
VN-OSK-095,2026-12-26,Purchased,7,Online order fulfillment
VN-OSK-110,2026-12-26,Purchased,7,Loyalty member transaction
NK-CTZ-110,2026-12-26,Sold,1,Loyalty member transaction
NK-DNK-115,2026-12-26,Purchased,6,Checked against PO #4821
AD-CLC-095,2026-12-27,Sold,1,Gift purchase with receipt
NK-DNK-130,2026-12-27,Sold,1,Customer exchange processed
AD-CLC-105,2026-12-27,Sold,3,Awaiting customer pickup
NK-CTZ-090,2026-12-27,Sold,2,Walk-in customer purchase
NK-DNK-100,2026-12-27,Sold,2,Holiday season restock
AD-CLC-105,2026-12-27,Sold,1,Weekend sale transaction
NK-AJ11-120,2026-12-27,Purchased,8,Phone order placed earlier
NK-DNK-100,2026-12-27,Purchased,6,Holiday season restock
NK-AJ11-110,2026-12-27,Purchased,10,Vendor delivery confirmed
AD-SMB-110,2026-12-28,Sold,1,Inventory audit adjustment
AD-CMP-120,2026-12-28,Purchased,12,Online order fulfillment
NK-CTZ-105,2026-12-28,Purchased,5,Verified by manager on duty
VN-OSK-095,2026-12-28,Purchased,5,Customer exchange processed
NK-DNK-100,2026-12-28,Purchased,12,Awaiting customer pickup
AD-SMB-110,2026-12-28,Sold,2,Return processed separately
NK-AF1-120,2026-12-29,Purchased,5,Walk-in customer purchase
NK-DNK-130,2026-12-29,Sold,1,Requested by floor supervisor
VN-OSK-090,2026-12-29,Sold,2,Online order fulfillment
NK-AF1-110,2026-12-29,Purchased,12,Walk-in customer purchase
NK-AJ11-110,2026-12-29,Sold,2,Staff discount applied
NK-DNK-110,2026-12-29,Sold,1,Holiday season restock
NK-AJ11-120,2026-12-29,Purchased,4,Customer exchange processed
AD-CMP-105,2026-12-29,Purchased,6,Customer exchange processed
VN-OSK-100,2026-12-29,Sold,2,Customer exchange processed
NK-DNK-090,2026-12-29,Sold,2,Special order item
NK-AJ11-120,2026-12-30,Purchased,10,Shipped from warehouse
NK-CTZ-090,2026-12-30,Purchased,9,Shipment arrived with minor box damage
AD-CLC-095,2026-12-30,Purchased,9,Weekend sale transaction
NK-AJ11-095,2026-12-30,Purchased,10,Rush order from supplier
NK-DNK-100,2026-12-30,Purchased,11,Gift purchase with receipt
NK-AF1-105,2026-12-30,Purchased,4,Holiday season restock
NK-AJ11-100,2026-12-30,Sold,1,Rush order from supplier
NK-CTZ-100,2026-12-30,Sold,3,Phone order placed earlier
NK-AF1-100,2026-12-30,Purchased,12,Walk-in customer purchase
AD-CMP-100,2026-12-30,Sold,1,Shipment arrived with minor box damage
VN-OSK-090,2026-12-30,Purchased,11,Gift purchase with receipt
NK-AF1-095,2026-12-30,Purchased,4,Customer exchange processed
AD-CMP-105,2026-12-30,Sold,3,Shipped from warehouse
AD-SMB-090,2026-12-30,Sold,3,Awaiting customer pickup
"""